<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/02_query_and_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# クエリ・フィルタ・トランザクション (在庫管理)

このノートブックは **pytidb シリーズの第 2 回** です。在庫管理を題材に、条件付き検索・並び替え・ページング・更新・トランザクションを扱います。

## 学習目標
- `filter` で演算子 (`$gt`, `$lt`, `$in` …) を使う
- `order_by` / `limit` / `offset` でページングする
- `table.update` / `table.delete` で一括更新
- `client.session()` でトランザクション (全件コミット or ロールバック)

前提: `00_quickstart.ipynb` を実行済み。


## 1. 依存パッケージのインストールとTiDB Cloudクラスタの作成


In [ ]:
!pip install -q pytidb


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-query")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 2. 在庫テーブルを作成

`sku`(SKU), `name`(商品名), `category`(カテゴリ), `stock`(在庫数), `price`(価格) の 5列で在庫テーブルを定義します。


In [ ]:
from pytidb import TiDBClient
from pytidb.schema import Field, TableModel

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="inventory_demo",
    ensure_db=True,
)


class Item(TableModel):
    __tablename__ = "inventory"
    __table_args__ = {"extend_existing": True}

    sku: str = Field(primary_key=True)
    name: str = Field()
    category: str = Field()
    stock: int = Field(default=0)
    price: float = Field(default=0.0)


table = db.create_table(schema=Item, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 3. データ投入


In [ ]:
SAMPLES = [
    Item(sku="A-001", name="ワイヤレスイヤホン",  category="electronics", stock=42, price=8980),
    Item(sku="A-002", name="スマートウォッチ",    category="electronics", stock=15, price=18900),
    Item(sku="A-003", name="USB-C ケーブル 1m",   category="electronics", stock=0,  price=890),
    Item(sku="B-001", name="エスプレッソ粉 200g", category="grocery",     stock=30, price=1280),
    Item(sku="B-002", name="抹茶パウダー 100g",   category="grocery",     stock=8,  price=2480),
    Item(sku="B-003", name="ダークチョコ 70%",    category="grocery",     stock=120, price=580),
    Item(sku="C-001", name="帆布トートバッグ",    category="fashion",     stock=22, price=3980),
    Item(sku="C-002", name="リネンシャツ",        category="fashion",     stock=0,  price=6900),
    Item(sku="C-003", name="ウールソックス",      category="fashion",     stock=65, price=1280),
]

table.bulk_insert(SAMPLES)
print(f"投入完了。{table.rows()} 件")


## 4. フィルタ式のバリエーション

`filters` は dict で、値に `{"$gt": X}` のような演算子を書けます。
利用できる演算子は: `$eq`, `$ne`, `$lt`, `$lte`, `$gt`, `$gte`, `$in`, `$nin` です。


In [ ]:
# 価格 5000 円以下のもの
cheap = table.query(filters={"price": {"$lte": 5000}}).to_pydantic()
print(f"[価格 5000 円以下] {len(cheap)} 件")

# electronics または fashion のどちらか
mixed = table.query(filters={"category": {"$in": ["electronics", "fashion"]}}).to_pydantic()
print(f"[electronics or fashion] {len(mixed)} 件")

# 在庫切れ
oos = table.query(filters={"stock": 0}).to_pydantic()
print(f"[在庫切れ] {[i.sku for i in oos]}")


## 5. 並び替えとページング

`order_by` に `{"カラム": "asc" | "desc"}` を渡します。`limit` / `offset` でページングもできます。


In [ ]:
# 在庫が少ない順に 3 件
tight = table.query(order_by={"stock": "asc"}, limit=3).to_pydantic()
print("[在庫少ない順 TOP3]")
for i in tight:
    print(f"  {i.sku} stock={i.stock} name={i.name}")

# 価格が高い順に 2 件ずつページング
print("\n[価格降順・2 件ずつ]")
for page in range(3):
    items = table.query(order_by={"price": "desc"}, limit=2, offset=page * 2).to_pydantic()
    print(f"  page {page}: {[i.name for i in items]}")


## 6. 一括更新と削除

条件付きで `update` / `delete` を行います。


In [ ]:
# grocery カテゴリだけ 10% 値上げ (生 SQL を使うと一括計算しやすい)
db.execute("UPDATE inventory SET price = price * 1.10 WHERE category = 'grocery'")

# 在庫ゼロの商品を削除
before = table.rows()
table.delete(filters={"stock": 0})
after = table.rows()
print(f"削除件数: {before - after}  (残り {after} 件)")


## 7. トランザクション (`client.session()`)

`with db.session()` を使いトランザクション処理を実行できます。ブロック内で例外が出れば自動ロールバック、問題なければコミットします。
次の例では、在庫の移動 (倉庫 A → 倉庫 B) を 2 つの UPDATE でアトミックに行います。


In [ ]:
# 例のため、カテゴリごとに在庫合計を出す関数を用意
def category_stock(cat):
    rows = db.query(
        "SELECT COALESCE(SUM(stock), 0) FROM inventory WHERE category = :cat",
        {"cat": cat},
    ).to_rows()
    return rows[0][0]

before_e = category_stock("electronics")
before_f = category_stock("fashion")
print(f"before: electronics={before_e}, fashion={before_f}")

MOVE_QTY = 5
with db.session():
    db.execute(
        "UPDATE inventory SET stock = stock - :q WHERE sku = 'A-002'",
        {"q": MOVE_QTY},
    )
    db.execute(
        "UPDATE inventory SET stock = stock + :q WHERE sku = 'C-003'",
        {"q": MOVE_QTY},
    )

after_e = category_stock("electronics")
after_f = category_stock("fashion")
print(f"after : electronics={after_e}, fashion={after_f}")
print("(electronics から fashion へ 5 個ぶん移動、合計は保存される)")


## 次のステップ

- `03_vector_search.ipynb` : 自然言語での意味類似検索
- `05_hybrid_search.ipynb` : ベクトル + 全文のハイブリッド

## チャレンジ

- `{"$and": [...], "$or": [...]}` の組み合わせを試す
- `db.query(sql, params)` で group by / having を書いてみる
- session 内で `raise Exception("abort")` したらロールバックされることを確認
